# 11 — Local-Level Embedding Clusters (K-Means per LA)

Repeats step 8's embedding K-Means independently for each Local Authority, weighting
respondents by how many times they appear in that LA's synthetic population (from step 10).

**Inputs:**

- `data/6_add_vector_embedding/k_vector_embedding.pkl` — `pidp`, `embedding`
- `data/10_synthetic_population/pidp_la_counts.csv` — `pidp`, `ladcd`, `ladnm`, `n`

**Output:** `data/11_cluster_embeddings_LA/la_embedding_clusters.csv`
— one row per `(pidp × ladcd)` with columns: `pidp`, `ladcd`, `ladnm`, `n`, `cluster`

**Config** (`config_cluster.py`):

- `N_CLUSTERS_LOCAL` — clusters per LA (default 5)
- `TEST_MODE` / `TEST_LA_CODES` — limit to a subset of LAs for testing


In [7]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_variables as _cv
importlib.reload(_cv)
from data_pipeline.config_variables import DATA_FOLDER

import data_pipeline.helpers.cluster as cf
importlib.reload(cf)

import data_pipeline.config_cluster as _cc
importlib.reload(_cc)
from data_pipeline.config_cluster import N_CLUSTERS_LOCAL, TEST_MODE, TEST_LA_CODES, WAVE

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import normalize

# ── Paths ─────────────────────────────────────────────────────────────────────
EMBED_PKL   = Path(f"../{DATA_FOLDER}/6_add_vector_embedding/k_vector_embedding.pkl")
FEATURE_PKL = Path(f"../{DATA_FOLDER}/4_feature_eng/{WAVE}_feature_eng.pkl")
LA_COUNTS   = Path(f"../{DATA_FOLDER}/10_synthetic_population/pidp_la_counts.csv")
OUT_DIR     = Path(f"../{DATA_FOLDER}/11_cluster_embeddings_LA")
OUT_CSV     = OUT_DIR / "la_embedding_clusters.csv"

for p in (EMBED_PKL, FEATURE_PKL, LA_COUNTS):
    if not p.exists():
        raise FileNotFoundError(f"{p} — check pipeline prerequisites.")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load embeddings ───────────────────────────────────────────────────────────
print("Loading embeddings …")
df_emb = pd.read_pickle(EMBED_PKL)[["pidp", "embedding"]].copy()
df_emb["pidp"] = pd.to_numeric(df_emb["pidp"], errors="coerce").astype("int64")

X_raw = np.array(df_emb["embedding"].tolist(), dtype=np.float32)
X_norm = normalize(X_raw, norm="l2", axis=1)
emb_index = df_emb["pidp"].values
pidp_to_row = {int(p): i for i, p in enumerate(emb_index)}
print(f"  {len(df_emb):,} respondents, embedding dim={X_norm.shape[1]}")

# ── Load LA counts ────────────────────────────────────────────────────────────
print("Loading LA counts …")
df_la = pd.read_csv(LA_COUNTS, dtype={"pidp": "int64", "ladcd": str, "ladnm": str, "n": "int64"})
print(f"  {len(df_la):,} (pidp × LA) rows, {df_la['ladcd'].nunique():,} LAs")

if TEST_MODE and TEST_LA_CODES:
    df_la = df_la[df_la["ladcd"].isin(TEST_LA_CODES)].copy()
    print(f"  TEST MODE — restricted to {df_la['ladcd'].nunique()} LA(s): "
          f"{sorted(df_la['ladnm'].unique().tolist())}")

all_la_codes = sorted(df_la["ladcd"].unique())
print(f"  Processing {len(all_la_codes)} LA(s) with N_CLUSTERS_LOCAL={N_CLUSTERS_LOCAL}")

# ── Cluster per LA ────────────────────────────────────────────────────────────
results = []

for i, ladcd in enumerate(all_la_codes, 1):
    la_rows = df_la[df_la["ladcd"] == ladcd].copy()
    ladnm   = la_rows["ladnm"].iloc[0]

    la_rows = la_rows[la_rows["pidp"].isin(pidp_to_row)]
    n_pidps = len(la_rows)

    if n_pidps < N_CLUSTERS_LOCAL:
        print(f"  [{i}/{len(all_la_codes)}] {ladnm} — only {n_pidps} pidps, skipping")
        continue

    row_indices = la_rows["pidp"].map(pidp_to_row).values
    X_la = X_norm[row_indices]
    w_la = la_rows["n"].values.astype(np.float64)

    k = min(N_CLUSTERS_LOCAL, n_pidps)
    labels = cf.fit_kmeans(X_la, k, sample_weight=w_la)

    la_rows = la_rows.copy()
    la_rows["cluster"] = labels + 1
    results.append(la_rows)

    counts_str = ", ".join(
        f"{v}" for v in sorted(pd.Series(labels + 1).value_counts().sort_index().values)
    )
    print(f"  [{i}/{len(all_la_codes)}] {ladnm} — {n_pidps} pidps → {k} clusters [{counts_str}]")

# ── Save raw pidp-level output ────────────────────────────────────────────────
df_out = pd.concat(results, ignore_index=True)
df_out.to_csv(OUT_CSV, index=False)
print(f"\nSaved {len(df_out):,} rows → {OUT_CSV}")

# ── Build demographic summary per (ladcd × cluster) ──────────────────────────
print("\nBuilding demographic summary …")
import data_pipeline.helpers.cluster_summary as _cs
importlib.reload(_cs)

df_feat = pd.read_pickle(FEATURE_PKL)
df_feat["pidp"] = pd.to_numeric(df_feat["pidp"], errors="coerce").astype("int64")

# Join cluster labels with feature demographics; n becomes n_sipher_rows for make_cluster_summary
df_merged = df_out.rename(columns={"n": "n_sipher_rows"}).merge(df_feat, on="pidp", how="left")

summary_rows = []
for ladcd_val, grp in df_merged.groupby("ladcd"):
    la_summary = _cs.make_cluster_summary(grp, "cluster", wave=WAVE)
    la_summary["ladcd"] = ladcd_val
    la_summary["ladnm"] = grp["ladnm"].iloc[0]
    summary_rows.append(la_summary)

df_summary = pd.concat(summary_rows, ignore_index=True)
df_summary = df_summary[
    ["ladcd", "ladnm"] + [c for c in df_summary.columns if c not in ("ladcd", "ladnm")]
]

# ── Copy summary to API folder ────────────────────────────────────────────────
_api_dir = Path("../api/data/clusters")
_api_dir.mkdir(parents=True, exist_ok=True)
_api_dest = _api_dir / "local_embedding_clusters.csv"
df_summary.to_csv(_api_dest, index=False)
print(f"Summary ({len(df_summary)} rows) → {_api_dest}")
print(df_summary[["ladcd", "ladnm", "cluster_id", "tribe_label", "n_respondents", "size"]].to_string(index=False))


Loading embeddings …
  27,330 respondents, embedding dim=1536
Loading LA counts …
  7,969,122 (pidp × LA) rows, 346 LAs
  TEST MODE — restricted to 4 LA(s): ['Hounslow', 'Islington', 'Newham', 'Tower Hamlets']
  Processing 4 LA(s) with N_CLUSTERS_LOCAL=5
  [1/4] Hounslow — 24869 pidps → 5 clusters [507, 2017, 2347, 9593, 10405]
  [2/4] Islington — 24792 pidps → 5 clusters [1520, 2744, 5442, 6030, 9056]
  [3/4] Newham — 22036 pidps → 5 clusters [1388, 1500, 1538, 8054, 9556]
  [4/4] Tower Hamlets — 23571 pidps → 5 clusters [509, 1936, 2333, 9262, 9531]

Saved 95,268 rows → ../data/11_cluster_embeddings_LA/la_embedding_clusters.csv

Building demographic summary …
Summary (20 rows) → ../api/data/clusters/local_embedding_clusters.csv
    ladcd         ladnm  cluster_id tribe_label  n_respondents  size
E09000018      Hounslow           1   Cluster 1           9593 38180
E09000018      Hounslow           2   Cluster 2            507 12937
E09000018      Hounslow           3   Cluster 3      